In [24]:
import os
import yaml
from rich import print

from langchain.chat_models import AzureChatOpenAI, ChatOpenAI
from langchain.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, PromptTemplate
from langchain.chains import SequentialChain, LLMChain

## Step 0: Front-camera view photo
Use `image_url` to import the photo you want to test. 

Feel free to try new photo with your own link!

In [62]:
from gradio_client import Client
from IPython.display import Image

image_url = "https://github.com/PJLab-ADG/DriveLikeAHuman/blob/main/assets/cones_on_truck_1.jpg?raw=true"
Image(url= image_url)


## Step 1: Use LLaMA-Adapter to generate a description of the photo

In [57]:
client = Client("http://llama-adapter.opengvlab.com/")
llama_result = client.predict(
    image_url,  # str representing input in 'Input' Image component
    "Describe the picture as details as possible and focus on the main object.Do not describe what you don't see. The objects in the picture are moving.",    
    512,  # int | float representing input in 'Max length' Slider component
    0.1,  # int | float representing input in 'Temperature' Slider component
    0.75,  # int | float representing input in 'Top p' Slider component
    fn_index=1
)
llama_result

Loaded as API: http://llama-adapter.opengvlab.com/ ✔


"The image features a white truck driving down a street, carrying a large load of traffic cones. The truck is filled with numerous traffic cones, which are stacked and secured in the back of the vehicle. The cones are of various sizes and are placed in different positions, covering the entire length of the truck. The scene captures the truck's journey, showcasing the impressive amount of traffic cones it is carrying."

## Step 2: Load  LLM and format LLaMA-Adapter output
The code support both OpenAI API or Azure OpenAI service. 

Set up your API key in `config.yaml`.

In [58]:
# 导入自定义LLM类
from HELLM import VolcanoDoubaoLLM

# 加载配置
VOLCANO_CONFIG = yaml.load(open('config.yaml'), Loader=yaml.FullLoader)

# 初始化Doubao-Seed-1.6模型
llm = VolcanoDoubaoLLM(
    api_key=VOLCANO_CONFIG['VOLCANO_API_KEY'],
    api_base=VOLCANO_CONFIG['VOLCANO_API_BASE'],
    model_name=VOLCANO_CONFIG['VOLCANO_MODEL']
)

In [59]:
human_message_prompt = HumanMessagePromptTemplate(
        prompt=PromptTemplate(
            template="""
            请将以下图片描述总结为几个关键点，用列表形式呈现：
            {llama_results}
            """,
            input_variables=["llama_results"],
        )
    )
chat_prompt_template = ChatPromptTemplate.from_messages([human_message_prompt])
chain = LLMChain(llm=llm, prompt=chat_prompt_template)

observation_result = chain.run(llama_result)
print(observation_result)

1. A white truck is driving down a street.
2. The truck is carrying a large load of traffic cones.
3. The cones are of various sizes and are stacked and secured in the back of the vehicle.
4. The cones cover the entire length of the truck.
5. The scene showcases the impressive amount of traffic cones the truck is carrying.

## Step 3: Find the unusual part of this driving scenario

In [60]:
observation_result += "\n ego车辆在后方行驶，并与前车保持适当距离。"

human_message_prompt = HumanMessagePromptTemplate(
    prompt=PromptTemplate(
        template="""
        你是一名驾驶助手，需要基于传感器观察结果分析当前驾驶场景中的异常情况。
        请仅根据提供的信息推理，不要假设未发生的危险。
        以下是传感器观察结果：
        ```{observation}```
        """,
        input_variables=["observation"],
    )
)
chat_prompt_template = ChatPromptTemplate.from_messages([human_message_prompt])
chain = LLMChain(llm=llm, prompt=chat_prompt_template)

abnormal_situation = chain.run(observation_result)
print(abnormal_situation)

Based on the given observation, there is nothing particularly unusual or worth noting in this driving scenario. The
vehicle `ego` is simply driving behind a white truck that is carrying a large load of traffic cones, which are 
stacked and secured in the back of the vehicle. The cones cover the entire length of the truck, and the scene 
showcases the impressive amount of traffic cones the truck is carrying. The vehicle `ego` is maintaining a proper 
distance from the truck, which is a safe driving practice.

## Step 4: Make the final decision

In [61]:
second_prompt = ChatPromptTemplate.from_template(
    """
    你是一名经验丰富的驾驶员，擅长处理复杂交通场景。请基于以下信息做出驾驶决策：
    1. 总结当前场景并判断是否危险。
    2. 回答ego车辆是否需要减速，并解释原因（40字以内）。
    
    异常情况：```{abnormal_text}```
    传感器观察：```{observation}```
    """
)
chain_two = LLMChain(llm=llm, prompt=second_prompt, output_key="analyze")
overall_simple_chain = SequentialChain(
    chains=[chain_two],
    input_variables=["abnormal_text", "observation"],
    output_variables=["analyze"],
)
response = overall_simple_chain({
    "abnormal_text": abnormal_situation, 
    "observation": observation_result
})
print(response["analyze"])

1. Summary: The current scenario involves a white truck carrying a large load of traffic cones, which cover the 
entire length of the truck. The vehicle 'ego' is driving behind the truck and maintaining a safe distance. There is
nothing particularly dangerous or unusual in this scenario.

2. No, the driver on 'ego' should not decide to decelerate the car because of the current situation. The vehicle 
'ego' is already maintaining a proper distance from the truck, which is a safe driving practice. Additionally, 
there is nothing particularly dangerous or unusual in this scenario that would require the driver to slow down.